### Kipf-Welling implementation (Full batch)

In [ ]:
# imports
from src.utils import *
from src.model import *
from src.train import *
from typing import Any
from dataclasses import dataclass
from sklearn.manifold import TSNE

import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import argparse
import math
import time
import numpy as np

In [ ]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
# run experiments for all 3 data sets; each for 9 iterations

def summarize_results(results, dataset_name):
    runs = results["run_summaries"]

    test_accs = [r["test_acc"] for r in runs]
        
    short_results = {
        "dataset": dataset_name,
        "test_acc_mean": np.mean(test_accs),
        "test_acc_std": np.std(test_accs, ddof=1),
        "best_val_acc_mean": np.mean([r["best_val_acc"] for r in runs]),
        "best_val_loss_mean": np.mean([r["best_val_loss"] for r in runs]),
        "total_train_time_mean": np.mean([r["total_train_time"] for r in runs]),
        "mean_epoch_train_time_mean": np.mean([r["mean_epoch_train_time"] for r in runs]),
    }
    return short_results

cfg_paths = [
    "configs/cora.yaml",
    "configs/pubmed.yaml",
    "configs/arxiv.yaml",
]

summaries = []

for path in cfg_paths:
    cfg = load_config(path)
    results = run_from_cfg(cfg)
    dataset_name = cfg["data"]["data_name"]
    summaries.append(summarize_results(results, dataset_name))


In [ ]:
# plotting







In [ ]:
# t-SNE

def run_single_for_viz(cfg):
    seeds = [int(s) for s in cfg["seeds"]]
    set_seed(seeds[0])

    device = torch.device(cfg["train"]["device"] if torch.cuda.is_available() else "cpu")

    data = load_data(
        cfg["data"]["data_name"],
        os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "data"),
        cfg["data"]["data_class"]
    )
    data_x = data.x.to(device)
    data_y = data.y.to(device)

    lr = float(cfg["train"]["lr"])
    epochs = int(cfg["train"]["epochs"])
    weight_decay = float(cfg["train"]["weight_decay"])
    hidden_dim = int(cfg["model"]["hidden_dim"])
    dropout = float(cfg["model"]["dropout"])
    early_stopping = int(cfg["model"]["early_stopping"])
    num_neighbors = cfg["data"]["num_neighbors"]
    batch_mode = cfg["data"]["batch_mode"]
    batch_size = int(cfg["data"]["batch_size"])

    input_dim = data.num_features
    output_dim = data.y.unique().size(0)

    train_cfg = TrainConfig(
        epochs=epochs,
        early_stop=early_stopping,
        batch_mode=batch_mode,
        batch_size=batch_size,
        num_neighbors=num_neighbors,
        device=device
    )

    graph_data = GraphDataBundle(
        x=data_x,
        y=data_y,
        A_hat=None,
        train_mask=data.train_mask,
        val_mask=data.val_mask,
        test_mask=data.test_mask,
        data=data
    )

    train_loader, val_loader, test_loader = None, None, None
    if batch_mode == "full_batch":
        graph_data.A_hat = compute_A_hat(data).to(device)
        model = GCNFullBatch(input_dim, hidden_dim, output_dim, dropout).to(device)
    else:
        train_loader, val_loader, test_loader = build_neighbor_loaders(graph_data, train_cfg)
        model = GCNMiniBatch(input_dim, hidden_dim, output_dim, dropout).to(device)

    optimizer = optim.Adam(
        [
            {"params": model.layer1.parameters(), "weight_decay": weight_decay},
            {"params": model.layer2.parameters(), "weight_decay": 0.0},
        ],
        lr=lr
    )
    loss_function = nn.CrossEntropyLoss()

    history, summary = train(model, optimizer, loss_function, graph_data, train_cfg, train_loader, val_loader)

    return model, graph_data, history, summary

cfg = load_config("configs/cora_1_seed.yaml")
model, graph_data, history, summary = run_single_for_viz(cfg)

model.eval()
with torch.no_grad():
    h = model.layer1(graph_data.x, graph_data.A_hat)   # full-batch case
    h = model.relu(h)

z = h.cpu().numpy()
y = graph_data.y.cpu().numpy()

z_2d = TSNE(n_components=2, random_state=42).fit_transform(z)

plt.figure(figsize=(8, 6))
plt.scatter(z_2d[:, 0], z_2d[:, 1], c=y, s=10)
plt.title("t-SNE of node embeddings")
plt.show()

### Addition: Mini-batch;;;;; rework

As we will see below, if ran for small graphs like Core, PubMed or Citeseer, the additional overhead (subgraph generation, sampling, many small instead of one not that much larger mat mul operatino) outweighs the performance gain.
But, nevertheless we still obtain a only little lower accuracy as with full batch.

Therefore we will use a second, much larger data set (ogbn-arxiv (100k nodes), maybe even ogbn-products (2 million nodes)); 

here we expect the full batch approach to be very slow or time out because of memory overflow
compare runtimes

In [ ]:
# mini-batch run of Cora
summaries_cmb = []
cfg = load_config("configs/cora_mini_batch.yaml")
results_cmb = run_from_cfg(cfg)
dataset_name = cfg["data"]["data_name"]
summaries_cmb.append(summarize_results(results_cmb, dataset_name))

In [ ]:
# full batch run of ogbn-arxiv
summaries_fb = []
cfg = load_config("configs/ogbn_arxiv_full_batch.yaml")
results_fb = run_from_cfg(cfg)
dataset_name = cfg["data"]["data_name"]
summaries_fb.append(summarize_results(results_fb, dataset_name))

# display runtimes

In [ ]:
# mini batch run of ogbn-arxiv
summaries_mb = []
cfg = load_config("configs/ogbn_arxiv_mini_batch.yaml")
results_mb = run_from_cfg(cfg)
dataset_name = cfg["data"]["data_name"]
summaries_mb.append(summarize_results(results_mb, dataset_name))

# display runtimes